<a href="https://colab.research.google.com/github/nissi31/fitness-pose-app/blob/main/FallDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Using yolo11s-pose

In [ ]:
print("Installing dependencies...")
!pip install -q ultralytics numpy opencv-python

import cv2
import json
import numpy as np
from ultralytics import YOLO

print("✅ Dependencies installed.")

try:
    model = YOLO('yolo11s-pose.pt')
    print("✅ YOLOv8-Pose model loaded successfully.")
except Exception as e:
    print(f"Error loading YOLO11s-Pose model: {e}")
    exit()

video_path = '/content/Human Fall Detection Sample.mp4'
output_video_path = 'fall_detection_output.mp4'
output_json_path = 'keypoints_data.json'

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, cap.get(cv2.CAP_PROP_FPS), (int(cap.get(3)), int(cap.get(4))))
print(f"✅ Output video will be saved to {output_video_path}")

previous_hip_y = {}
FALL_VELOCITY_THRESHOLD = 15

keypoint_data = []
frame_idx = 0

print("\nProcessing video... This may take a moment.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    results = model.track(frame, persist=True)

    annotated_frame = results[0].plot()

    if results[0].boxes.id is not None:
        track_ids = results[0].boxes.id.int().cpu().tolist()
        boxes = results[0].boxes.xywh.cpu().numpy()
        all_keypoints = results[0].keypoints.xy.cpu().numpy()

        for i, track_id in enumerate(track_ids):
            kpts = all_keypoints[i]

            left_hip_y = kpts[11][1]
            right_hip_y = kpts[12][1]

            if kpts[11][0] > 0 and kpts[12][0] > 0:
                current_hip_y = (left_hip_y + right_hip_y) / 2

                if track_id in previous_hip_y:
                    vertical_velocity = current_hip_y - previous_hip_y[track_id]

                    if vertical_velocity > FALL_VELOCITY_THRESHOLD:
                        box_w, box_h = boxes[i][2], boxes[i][3]
                        if box_w > box_h:
                            box_x1, box_y1 = int(boxes[i][0] - box_w/2), int(boxes[i][1] - box_h/2)
                            cv2.putText(
                                annotated_frame,
                                "FALL DETECTED!",
                                (box_x1, box_y1 - 10),
                                cv2.FONT_HERSHEY_DUPLEX,
                                1, (0, 0, 255), 2
                            )

                previous_hip_y[track_id] = current_hip_y

    out.write(annotated_frame)

    if results[0].keypoints is not None:
        for kpts in results[0].keypoints.data:
            keypoint_data.append({
                "frame": frame_idx,
                "keypoints": kpts.cpu().numpy().tolist()
            })

    if frame_idx % 100 == 0:
        print(f"Processed frame {frame_idx}...")
    frame_idx += 1

cap.release()
out.release()
print("\n✅ Video processing complete!")

with open(output_json_path, 'w') as f:
    json.dump(keypoint_data, f, indent=4)

print(f"Keypoint data saved to {output_json_path}")
print(f"Annotated video with fall detection saved to {output_video_path}")

Installing dependencies...
✅ Dependencies installed.
✅ YOLOv8-Pose model loaded successfully.
✅ Output video will be saved to fall_detection_output.mp4

Processing video... This may take a moment.

0: 384x640 1 person, 15.6ms
Speed: 1.6ms preprocess, 15.6ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Processed frame 0...

0: 384x640 1 person, 12.0ms
Speed: 1.9ms preprocess, 12.0ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 12.0ms
Speed: 1.7ms preprocess, 12.0ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 12.0ms
Speed: 1.9ms preprocess, 12.0ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 14.5ms
Speed: 2.2ms preprocess, 14.5ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 12.1ms
Speed: 1.8ms preprocess, 12.1ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 12.0

#Using yolov8n-pose

In [ ]:
!pip install -q ultralytics numpy opencv-python

import cv2
from ultralytics import YOLO
import json
import numpy as np

model = YOLO('yolov8n-pose.pt')

video_path = '/content/Human Fall Detection Sample.mp4'
cap = cv2.VideoCapture(video_path)

previous_hip_y = {}
FALL_VELOCITY_THRESHOLD = 15

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('fall_detection_output.mp4', fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))

keypoint_data = []
frame_idx = 0

print("Processing video... This may take a moment.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True)

    annotated_frame = results[0].plot()

    if results[0].boxes.id is not None:
        track_ids = results[0].boxes.id.int().cpu().tolist()
        boxes = results[0].boxes.xywh.cpu().numpy()

        all_keypoints = results[0].keypoints.xy.cpu().numpy()

        for i, track_id in enumerate(track_ids):
            kpts = all_keypoints[i]

            left_hip_y = kpts[11][1]
            right_hip_y = kpts[12][1]

            current_hip_y = (left_hip_y + right_hip_y) / 2

            fall_detected_this_frame = False

            if track_id in previous_hip_y:
                vertical_velocity = current_hip_y - previous_hip_y[track_id]

                if vertical_velocity > FALL_VELOCITY_THRESHOLD:
                    box_w, box_h = boxes[i][2], boxes[i][3]
                    if box_w > box_h:
                        fall_detected_this_frame = True

                        box_x1, box_y1 = int(boxes[i][0] - box_w/2), int(boxes[i][1] - box_h/2)

                        cv2.putText(
                            annotated_frame,
                            "FALL DETECTED!",
                            (box_x1, box_y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1,
                            (0, 0, 255),
                            2
                        )

            previous_hip_y[track_id] = current_hip_y

    out.write(annotated_frame)

    for result in results:
        if result.keypoints is not None:
            for kpts in result.keypoints.xy:
                frame_data = {
                    "frame": frame_idx,
                    "keypoints": kpts.tolist()
                }
                keypoint_data.append(frame_data)

    frame_idx += 1

cap.release()
out.release()

with open('keypoints_data.json', 'w') as f:
    json.dump(keypoint_data, f, indent=4)

print("✅ Processing complete!")
print("Keypoint data saved to keypoints_data.json")
print("Annotated video with fall detection saved to fall_detection_output.mp4")

100%|██████████| 6.52M/6.52M [00:00<00:00, 373MB/s]

Processing video... This may take a moment.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...



requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


0: 384x640 1 person, 37.5ms
Speed: 1.4ms preprocess, 37.5ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8.0ms
Speed: 2.0ms preprocess, 8.0ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8.1ms
Speed: 1.6ms preprocess, 8.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8.4ms
Speed: 2.2ms preprocess, 8.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 15.8ms
Speed: 1.8ms preprocess, 15.8ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 7.9ms
Speed: 1.5ms preprocess, 7.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 7.8ms
Speed: 1.6ms preprocess, 7.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)



#Using mediapipe
MediaPipe was better it caught all the falls in the video

In [ ]:
print("Installing dependencies...")
!pip install -q mediapipe opencv-python

import cv2
import mediapipe as mp
import numpy as np
import os
import logging
import time

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

print("✅ Dependencies installed and MediaPipe initialized.")

class PoseDetector:
    def __init__(self, confidence_threshold: float = 0.5):
        try:
            self.pose = mp_pose.Pose(
                static_image_mode=False,
                model_complexity=1, # 0=light, 1=full, 2=heavy
                min_detection_confidence=confidence_threshold,
                min_tracking_confidence=confidence_threshold
            )
            logger.info("MediaPipe Pose model initialized successfully.")
        except Exception as e:
            logger.error(f"Failed to initialize MediaPipe: {e}")
            raise

    def process_video(self, input_path: str, output_path: str, fall_velocity_threshold: float = 8.0):
        try:
            cap = cv2.VideoCapture(input_path)
            if not cap.isOpened():
                logger.error(f"Cannot open video: {input_path}")
                return False

            fps = cap.get(cv2.CAP_PROP_FPS)
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
            if not out.isOpened():
                logger.error(f"Failed to initialize VideoWriter for {output_path}")
                cap.release()
                return False

            prev_hip_y = None
            fall_detected_timer = 0
            frame_count = 0
            start_time = time.time()
            logger.info(f"Processing video: {input_path}")
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = self.pose.process(rgb_frame)
                is_fall = False
                if results.pose_landmarks:
                    landmarks = results.pose_landmarks.landmark
                    left_hip_y = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y * height
                    right_hip_y = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y * height
                    current_hip_y = (left_hip_y + right_hip_y) / 2
                    if prev_hip_y is not None:
                        vertical_velocity = current_hip_y - prev_hip_y
                        if vertical_velocity > fall_velocity_threshold:
                            is_fall = True
                            fall_detected_timer = int(fps)

                    prev_hip_y = current_hip_y

                mp_drawing.draw_landmarks(
                    frame,
                    results.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    landmark_drawing_spec=mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
                    connection_drawing_spec=mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2)
                )
                if fall_detected_timer > 0:
                    cv2.putText(frame, "FALL DETECTED!", (50, 50), cv2.FONT_HERSHEY_DUPLEX,
                                1.5, (0, 0, 255), 2)
                    fall_detected_timer -= 1
                out.write(frame)
                frame_count += 1
                if frame_count % 100 == 0:
                    logger.info(f"Processed frame {frame_count}...")

            cap.release()
            out.release()
            self.pose.close()

            logger.info(f"Video processing complete. Output saved to {output_path}")
            return True

        except Exception as e:
            logger.error(f"An error occurred during video processing: {e}")
            return False

if __name__ == "__main__":
    input_video = '/content/Human Fall Detection Sample.mp4'
    output_video = 'mediapipe_fall_detection_output.mp4'

    if not os.path.exists(input_video):
        logger.error(f"Input file not found: {input_video}")
        logger.error("Please upload 'Human Fall Detection Sample.mp4' to your Colab session.")
    else:
        detector = PoseDetector(confidence_threshold=0.5)
        success = detector.process_video(input_video, output_video)

        if success:
            print("\n✅ Processing finished successfully!")
        else:
            print("\n❌ Processing failed. Check logs for errors.")

Installing dependencies...
✅ Dependencies installed and MediaPipe initialized.

✅ Processing finished successfully!
